<a href="https://colab.research.google.com/github/RVC-Boss/GPT-SoVITS/blob/main/Colab-WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phải connect drive trước
vì add dấu câu nó mặc định xuất model vào drive nên cần phải kết nối trước

In [1]:
# kết nối để lưu model khi train, đồng thời lấy model đó lên để sử dụng khi infer
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# chatbot cài thư viện

In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 122.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
!pip install -U lightrag-hku "flask[async]" pyngrok tqdm langchain-community langchain-text-splitters pipmaster pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requ

# cài đặt,khởi chạy ollamma

In [4]:
# 1. Cài đặt gói zstd để script Ollama có thể giải nén
!apt-get update && apt-get install -y zstd

# 2. Tiến hành cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh





Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restric

In [18]:
# 3. Khởi chạy Ollama ngầm (Background)
import subprocess
import time

# Chạy ollama serve và chuyển hướng log để không làm nghẽn hiển thị của Colab
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

time.sleep(10) # Tăng lên 10 giây để đảm bảo Ollama đã khởi động hoàn toàn trong background

# tải model về bằng ollama

In [19]:
# 4. Kéo các model về Colab
!ollama pull embeddinggemma
# !ollama pull gemma4:e4b
!ollama pull qwen2.5:7b
!ollama list



NAME                     ID              SIZE      MODIFIED               
qwen2.5:7b               845dbda0ea48    4.7 GB    Less than a second ago    
embeddinggemma:latest    85462619ee72    621 MB    1 second ago              
qwen3.5:9b               6488c96fa5fa    6.6 GB    18 minutes ago            


# thay thế lightrag và source

In [52]:
!rm -rf /content/chatbot
!git clone https://github.com/CongMinh-Dev/chatbot.git

Cloning into 'chatbot'...
remote: Enumerating objects: 504, done.
remote: Counting objects: 100% (504/504), done.
remote: Compressing objects: 100% (382/382), done.
remote: Total 504 (delta 114), reused 478 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (504/504), 7.61 MiB | 22.08 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [53]:
!rm -rf /usr/local/lib/python3.12/dist-packages/lightrag
!cp -r /content/chatbot/.venv/Lib/site-packages/lightrag/ /usr/local/lib/python3.12/dist-packages/lightrag

# Xóa lightrag_db

In [54]:
%cd /content
!rm -rf /content/chatbot/lightrag_db
!rm -rf /content/chatbot/lightrag_snapshots

/content


# **liên kết thư mục**

In [55]:
!rm -rf /content/chatbot/lightrag_snapshots/*

In [56]:
!rm -rf /content/chatbot/lightrag_snapshots

In [57]:
# Tạo liên kết:
!ln -s /content/drive/MyDrive/AI_Share/chatbot/lightrag_snapshots /content/chatbot

# chạy app

In [58]:
# Khởi chạy lại Ollama ngầm (Background), vì mỗi lần tắt dừng vì lỗi thì nó dừng ollama luôn
import subprocess
import time

# Chạy ollama serve và chuyển hướng log để không làm nghẽn hiển thị của Colab
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

time.sleep(10) # Tăng lên 10 giây để đảm bảo Ollama đã khởi động hoàn toàn trong background

In [59]:
!python /content/chatbot/ingest.py

/content/chatbot/ingest.py:24: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader
/content/chatbot/ingest.py:123: DeprecationWarning: There is no current event loop
  loop = asyncio.get_event_loop()
🧠 Đang khởi tạo LightRAG kết nối Ollama...
INFO: [] Created new empty graph file: /content/chatbot/lightrag_db/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '/content/chatbot/lightrag_db/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '/content/chatbot/lightrag_db/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': '/content/chatbot/lightr